### 1️⃣ Transformer 整体结构概览


Transformer 的原始论文（Vaswani et al., 2017）提出了 Encoder-Decoder 架构，核心特点：

完全基于注意力机制（Attention），不再使用 RNN/CNN。

高度并行化，适合大规模训练。

核心思想：每个位置的表示都可以与序列中所有位置进行交互。

### 2️⃣ Transformer 的核心模块

#### Self-Attention（自注意力）

目的：让序列中每个 token 可以关注其他所有 token，捕捉全局依赖。

$$ 
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

Q（Query）：表示“我想找什么”

𝐾（Key）：表示“这里有什么”

𝑉（Value）：表示“实际信息是什么”

$d_k$ 是 Key 的维度，用来做缩放，防止数值过大

#### Multi-Head Attention（多头注意力）

问题：单个注意力头可能只能捕捉单一的关系模式。

做法：同时使用多个注意力头，每个头学不同的特征子空间，最后拼接输出。

$$ 
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O
$$

$$ 
\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)
$$

#### Positional Encoding（位置编码）

问题：Transformer 没有 RNN 的顺序感知能力，无法知道 token 的位置信息。

做法：在输入 embedding 上加上固定或可学习的位置向量。

$$
PE_{(pos,2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$
$$
PE_{(pos,2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

#### Feed-Forward Network（前馈网络）

每个 Encoder/Decoder 层有一个两层全连接网络

对每个 token 独立处理（位置无关）

$$
\text{FFN}(x) = \text{max}(0, x W_1 + b_1) W_2 + b_2

$$

#### Residual + LayerNorm（残差 + 层归一化）

问题：深层堆叠 Attention 和 FFN，梯度容易消失/爆炸。

做法：

加残差连接：output = x + Module(x)

再做 LayerNorm，稳定训练

#### Decoder 的 Mask（掩码）

问题：在序列生成时，预测 token 时不能看到未来 token。

做法：

Mask 掩掉未来位置

Attention 权重矩阵的上三角置为负无穷 → softmax 后为 0

### 先来一个简单的demo理解self-attention的计算方式

In [1]:
import torch

x = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 1.0, 1.0]
])

d_k = 4
W_Q = torch.eye(4) # 单位阵 (4, 4)
W_K = torch.eye(4)
W_V = torch.eye(4)

Q = x @ W_Q  # (3, 4) @ (4, 4) -> (3, 4)
K = x @ W_K
V = x @ W_V

# 计算attention
scores = Q @ K.T / torch.sqrt(torch.tensor(d_k, dtype=torch.float32)) # (3, 3)

weights = torch.softmax(scores, dim=-1) # 每行表示该token对其他token的注意力，经此调整和为1

output = weights @ V # (3, 3) @ (3, 4) -> (3, 4)

### 再来看看multi-head attention

In [ ]:
import torch

# 输入序列
x = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 2.0, 0.0, 2.0],
    [1.0, 1.0, 1.0, 1.0]
])

h = 2  # head数量
d_model = 4
d_k = d_model // h  # 每个头的维度

W_Q = torch.eye(d_model)
W_K = torch.eye(d_model)
W_V = torch.eye(d_model)
W_O = torch.eye(d_model)


Q1, K1, V1 = x[:, :2], x[:, :2], x[:, :2]
Q2, K2, V2 = x[:, 2:], x[:, 2:], x[:, 2:]


import torch.nn.functional as F

def attention(Q, K, V):
    scores = Q @ K.T / torch.sqrt(torch.tensor(Q.shape[1], dtype=torch.float32))
    weights = F.softmax(scores, dim=-1)
    return weights @ V

head1_out = attention(Q1, K1, V1)
head2_out = attention(Q2, K2, V2)
multihead_out = torch.cat([head1_out, head2_out], dim=-1)
multihead_out = multihead_out @ W_O

# 不好理解，重新认识

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

test = True
def djpprint(*args, **kwargs):
    if test:
        print(*args, **kwargs)
    return

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        batch_size, seq_len, d_model = x.size()
        
        # 分成多头
        # W_Q: (512 -> 512), x: (32, 50, 512)
        djpprint("x.shape", x.shape)
        djpprint("W_Q(x).shape", self.W_Q(x).shape)
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        djpprint("self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).shape", self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).shape)  # (32, 50, 8, 64)
        djpprint("Q.shape", Q.shape)  # (32, 8, 50, 64)
        
        # 计算注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        weights = F.softmax(scores, dim=-1) 
        weights = self.dropout(weights)
        out = torch.matmul(weights, V)
        djpprint("scores.shape", scores.shape)  # (32, 8, 50, 50)
        djpprint(scores[0, 0, 0, :])
        djpprint(F.softmax(scores[0, 0, 0, :]))
        djpprint(F.softmax(scores, dim=-1)[0, 0, 0, :])
        
        # 拼接多头
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        out = self.W_O(out)
        return out

class PositionwiseFeedForwar1d(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))
    

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.norm1(x + self.dropout(self.self_attn(x)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


x = torch.randn(32, 50, 512)
layer = TransformerEncoderLayer()
y = layer(x)
print(y.shape)

x.shape torch.Size([32, 50, 512])
W_Q(x).shape torch.Size([32, 50, 512])
self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).shape torch.Size([32, 50, 8, 64])
Q.shape torch.Size([32, 8, 50, 64])
scores.shape torch.Size([32, 8, 50, 50])
tensor([-0.0601, -0.4984,  0.5799, -0.1516,  0.5259, -0.2024,  0.3554, -0.2623,
         0.2940, -0.0300,  0.0435,  0.5502, -0.0694, -0.1616, -0.2597,  0.0288,
        -0.4525, -0.4550,  0.1191,  0.1527,  0.1035, -0.1623, -0.6872,  0.3613,
         0.2977, -0.4705, -0.3198, -0.0013,  0.3805,  0.2960, -0.1685, -0.2209,
        -0.1231,  0.2794,  0.1037, -0.4411,  0.4703,  0.1637, -0.3072, -0.1593,
        -0.2773, -0.4888,  0.2143,  0.5047, -0.3718, -0.0036,  0.2000,  0.1459,
        -0.1835, -0.3022], grad_fn=<SliceBackward0>)
tensor([0.0183, 0.0118, 0.0347, 0.0167, 0.0329, 0.0159, 0.0278, 0.0150, 0.0261,
        0.0189, 0.0203, 0.0337, 0.0181, 0.0165, 0.0150, 0.0200, 0.0124, 0.0123,
        0.0219, 0.0227, 0.0216, 0.0165, 0.0098, 0.0279, 0.0

/tmp/ipykernel_241676/570026662.py:44: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  djpprint(F.softmax(scores[0, 0, 0, :]))


#### 位置编码

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

In [22]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x
        

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TransformerEncoderLayer(nn.Module):
    def __init__(self,d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.norm1(x + self.dropout(self.self_attn(x)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x
    

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, d_ff=2048, num_layers=6,max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

## 完整的transformer encoder


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

test = True
def djpprint(*args, **kwargs):
    if test:
        print(*args, **kwargs)
    return

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        batch_size, seq_len, d_model = x.size()
        
        # 分成多头
        # W_Q: (512 -> 512), x: (32, 50, 512)
        djpprint("x.shape", x.shape)
        djpprint("W_Q(x).shape", self.W_Q(x).shape)
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        djpprint("self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).shape", self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).shape)  # (32, 50, 8, 64)
        djpprint("Q.shape", Q.shape)  # (32, 8, 50, 64)
        
        # 计算注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        weights = F.softmax(scores, dim=-1) 
        weights = self.dropout(weights)
        out = torch.matmul(weights, V)
        djpprint("scores.shape", scores.shape)  # (32, 8, 50, 50)
        djpprint(scores[0, 0, 0, :])
        djpprint(F.softmax(scores[0, 0, 0, :]))
        djpprint(F.softmax(scores, dim=-1)[0, 0, 0, :])
        
        # 拼接多头
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        out = self.W_O(out)
        return out

class PositionwiseFeedForwar1d(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))
    
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x
    
    
class TransformerEncoderLayer(nn.Module):
    def __init__(self,d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        x = self.norm1(x + self.dropout(self.self_attn(x)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x
    

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, d_ff=2048, num_layers=6,max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

## transformer decoder


#### Masked Self-Attention

$$ 
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + Mask\right) V
$$

Mask.shape = (seq_len, seq_len)

j <= i 时, Mask[i, j] = 0

j >  i 时, Mask[i, j] = 负无穷

In [10]:
# 实现mask函数
def generate_subsequent_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf')).masked_fill(mask == 0, float(0.0))
    return mask

### 开始实现decoder

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def generate_subsequent_mask(seq_len, device=None):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf')).masked_fill(mask == 0, float(0.0))
    return mask

# 可选
def create_padding_mask(pad_mask):
    if pad_mask is None:
        return None
    additive = pad_mask.unsqueeze(1).unsqueeze(1).to(torch.float32) * float('-inf')
    return additive

# 通用 MultiHeadAttention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, attn_mask=None):
        batch_size, tgt_len, _ = query.size()
        _, src_len, _ = key.size()
        
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)
        
        Q = Q.view(batch_size, tgt_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, src_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, src_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if attn_mask is not None:
            scores = scores + attn_mask
        
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = torch.matmul(weights, V)
        
        out = out.transpose(1, 2).contiguous().view(batch_size, tgt_len, self.d_model)
        out = self.W_O(out)
        return out, weights
    
# FFN
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))
    
# decoder layer
class TransformerDecoderLayer(nn.Module): 
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        """_summary_

        Args:
            tgt (_type_): (batch, tgt_len, d_model)  -- decoder input embeddings
            memory (_type_): (batch, src_len, d_model)  -- encoder outputs
            tgt_mask (_type_, optional): _description_. Defaults to None.
            memory_mask (_type_, optional): _description_. Defaults to None.
        """
        _tgt, self_w = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask)
        tgt = self.norm1(tgt + self.dropout(_tgt))
        
        _tgt2, cross_w = self.cross_attn(tgt, memory, memory, attn_mask=memory_mask)
        tgt = self.norm2(tgt + self.dropout(_tgt2))
        
        _tgt3 = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout(_tgt3))
        
        return tgt, self_w, cross_w
    
# decoder
class TransformerDecoder(nn.Module):
    def __init__(self, num_layers=6, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        attn_weights_self =[]
        attn_weights_cross = []
        x = tgt
        for layer in self.layers:
            x, w_self, w_cross = layer(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)
            attn_weights_self.append(w_self)
            attn_weights_cross.append(w_cross)
        x = self.norm(x)
        return x, attn_weights_self, attn_weights_cross


# test
batch = 2
src_len = 7
tgt_len = 5
d_model = 32
heads = 4

memory = torch.randn(batch, src_len, d_model)
tgt = torch.randn(batch, tgt_len, d_model)
decoder = TransformerDecoder(num_layers=2, d_model=d_model, num_heads=heads, d_ff=64)

casual = generate_subsequent_mask(tgt_len)
casual = casual.unsqueeze(0).unsqueeze(0)

out, attn_self, attn_cross = decoder(tgt, memory, tgt_mask=casual, memory_mask=None)

print("out.shape:", out.shape)  # (batch, tgt_len, d_model) = (2, 5, 32)
print("self-attn weights (layer0) shape:", attn_self[0].shape)   # (batch, num_heads, tgt_len, tgt_len)
print("cross-attn weights (layer0) shape:", attn_cross[0].shape) # (batch, num_heads, tgt_len, src_len)


out.shape: torch.Size([2, 5, 32])
self-attn weights (layer0) shape: torch.Size([2, 4, 5, 5])
cross-attn weights (layer0) shape: torch.Size([2, 4, 5, 7])


## 使用通用attention重写encoder

In [14]:
class PositionwiseFeedForwar1d(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))
    
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x
    
    
class TransformerEncoderLayer(nn.Module):
    def __init__(self,d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x
    

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, d_ff=2048, num_layers=6,max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

## 合成transformer

In [15]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, dropout=0.1, max_len=5000):
        super().__init__()
        self.encoder = TransformerEncoder(src_vocab_size, d_model, num_heads, d_ff, num_encoder_layers, max_len)
        self.decoder = TransformerDecoder(num_decoder_layers, d_model, num_heads, d_ff, dropout)
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        memory = self.encoder(src)
        out, attn_self, attn_cross = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_mask=None)
        logits = self.output_linear(out)
        
        return logits, attn_self, attn_cross
    

## 测试

In [16]:
src_vocab_size = 1000
tgt_vocab_size = 1000
batch_size = 2
src_len = 7
tgt_len = 5

model = Transformer(src_vocab_size, tgt_vocab_size, d_model=32, num_heads=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=64)

# 准备mask
src = torch.randint(0, src_vocab_size, (batch_size, src_len))
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_len))

tgt_mask = generate_subsequent_mask(tgt_len).unsqueeze(0).unsqueeze(0)

logits, attn_self, attn_cross = model(src, tgt, tgt_mask=tgt_mask)

print("logits.shape: ", logits.shape)

ValueError: not enough values to unpack (expected 3, got 2)

## final transformer

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 通用 MultiHeadAttention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, attn_mask=None):
        batch_size, tgt_len, _ = query.size()
        _, src_len, _ = key.size()
        
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)
        
        Q = Q.view(batch_size, tgt_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, src_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, src_len, self.num_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if attn_mask is not None:
            scores = scores + attn_mask
        
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = torch.matmul(weights, V)
        
        out = out.transpose(1, 2).contiguous().view(batch_size, tgt_len, self.d_model)
        out = self.W_O(out)
        return out, weights

# ------------------------encoder--------------------------#
class PositionwiseFeedForwar1d(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))
    
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x
    
    
class TransformerEncoderLayer(nn.Module):
    def __init__(self,d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x
    

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_heads=8, d_ff=2048, num_layers=6,max_len=5000):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)
    
#---------------------------decoder-----------------------------#
def generate_subsequent_mask(seq_len, device=None):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf')).masked_fill(mask == 0, float(0.0))
    return mask

# 可选
def create_padding_mask(pad_mask):
    if pad_mask is None:
        return None
    additive = pad_mask.unsqueeze(1).unsqueeze(1).to(torch.float32) * float('-inf')
    return additive
    
# FFN
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))
    
# decoder layer
class TransformerDecoderLayer(nn.Module): 
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        """_summary_

        Args:
            tgt (_type_): (batch, tgt_len, d_model)  -- decoder input embeddings
            memory (_type_): (batch, src_len, d_model)  -- encoder outputs
            tgt_mask (_type_, optional): _description_. Defaults to None.
            memory_mask (_type_, optional): _description_. Defaults to None.
        """
        _tgt, self_w = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask)
        tgt = self.norm1(tgt + self.dropout(_tgt))
        
        _tgt2, cross_w = self.cross_attn(tgt, memory, memory, attn_mask=memory_mask)
        tgt = self.norm2(tgt + self.dropout(_tgt2))
        
        _tgt3 = self.ffn(tgt)
        tgt = self.norm3(tgt + self.dropout(_tgt3))
        
        return tgt, self_w, cross_w
    
# decoder
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, num_layers=6, d_model=512, num_heads=8, d_ff=2048, dropout=0.1, max_len=5000):
        super().__init__()
        # 要加embedding，否则tgt少一个维度
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        x = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)
        x = self.pos_encoding(x)
        
        attn_weights_self =[]
        attn_weights_cross = []
        
        for layer in self.layers:
            x, w_self, w_cross = layer(x, memory, tgt_mask=tgt_mask, memory_mask=memory_mask)
            attn_weights_self.append(w_self)
            attn_weights_cross.append(w_cross)
        x = self.norm(x)
        return x, attn_weights_self, attn_weights_cross
    
# -------------------------合成transformer------------------------#
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, num_encoder_layers=6, num_decoder_layers=6, d_ff=2048, dropout=0.1, max_len=5000):
        super().__init__()
        self.encoder = TransformerEncoder(src_vocab_size, d_model, num_heads, d_ff, num_encoder_layers, max_len)
        self.decoder = TransformerDecoder(tgt_vocab_size, num_decoder_layers, d_model, num_heads, d_ff, dropout, max_len)
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)
        
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        memory = self.encoder(src)
        out, attn_self, attn_cross = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_mask=None)
        logits = self.output_linear(out)
        
        return logits, attn_self, attn_cross
    
# ------------------------测试----------------------------#
src_vocab_size = 1000
tgt_vocab_size = 1000
batch_size = 2
src_len = 7
tgt_len = 5

model = Transformer(src_vocab_size, tgt_vocab_size, d_model=32, num_heads=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=64)

# 准备mask
src = torch.randint(0, src_vocab_size, (batch_size, src_len))
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_len))

tgt_mask = generate_subsequent_mask(tgt_len).unsqueeze(0).unsqueeze(0)

logits, attn_self, attn_cross = model(src, tgt, tgt_mask=tgt_mask)

print("logits.shape: ", logits.shape)

logits.shape:  torch.Size([2, 5, 1000])
